In [10]:
# imports

import os
import sys

sys.path.append(os.path.abspath(r"C:\Users\sailj\OneDrive\文档\GitHub\Ad Conversion Model\src"))

from preprocessing import get_X_and_y
import pandas as pd
from sklearn.model_selection import train_test_split , RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score
from scipy.stats import randint , uniform

In [18]:
# 1st try

df = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\Ad Conversion Model\dataset\digital_marketing_campaign_dataset.csv")
X , y = get_X_and_y(df)

cat_cols = X.select_dtypes(include=["object" , "string"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

weights = compute_sample_weight(
    class_weight="balanced" ,
    y = y_train
)

preprocessing = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
    ("num" , StandardScaler() , num_cols)
])

pipeline = Pipeline([
    ("preprocess" , preprocessing) ,
    ("xgbc" , XGBClassifier(
        n_jobs = -1,
        random_state = 42 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8 ,
    ))
])

param_grid = {
    "xgbc__n_estimators" : randint(200 , 401) ,
    "xgbc__max_depth" : randint(3 , 7) ,
    "xgbc__learning_rate" : uniform(0.01 , 0.29),
    "xgbc__gamma" : uniform(0,5), 
    "xgbc__reg_lambda" : uniform(0,30),
    "xgbc__reg_alpha" : uniform(0,10),
    "xgbc__min_child_weight" : [1,2,3]
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid ,
    cv = 5 ,
    n_iter = 15
)

srch.fit(
    X_train ,
    y_train ,
    xgbc__sample_weight = weights
)

model = srch.best_estimator_

print(srch.best_params_)

y_pred = model.predict(X_test)
print(f"F1 SCORE : {f1_score(y_test , y_pred)}")

{'xgbc__gamma': np.float64(0.026451766267631127), 'xgbc__learning_rate': np.float64(0.2118632741240686), 'xgbc__max_depth': 5, 'xgbc__min_child_weight': 3, 'xgbc__n_estimators': 374, 'xgbc__reg_alpha': np.float64(5.178243981172603), 'xgbc__reg_lambda': np.float64(26.39575473174006)}
F1 SCORE : 0.9428989751098097


# Considering, n_estimators = 374 , max_depth = 5 as base line and changing range of other parameters accordingly

In [19]:
# 2nd Try

df = pd.read_csv(r"C:\Users\sailj\OneDrive\文档\GitHub\Ad Conversion Model\dataset\digital_marketing_campaign_dataset.csv")
X , y = get_X_and_y(df)

cat_cols = X.select_dtypes(include=["object" , "string"]).columns
num_cols = X.select_dtypes(include=["number"]).columns

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state=42 , stratify=y)

weights = compute_sample_weight(
    class_weight="balanced" ,
    y = y_train
)

preprocessing = ColumnTransformer([
    ("cat" , OneHotEncoder(handle_unknown="ignore") , cat_cols) ,
    ("num" , StandardScaler() , num_cols)
])

pipeline = Pipeline([
    ("preprocess" , preprocessing) ,
    ("xgbc" , XGBClassifier(
        n_jobs = -1,
        random_state = 42 ,
        subsample = 0.8 ,
        colsample_bytree = 0.8 ,
        n_estimators = 374 ,
        max_depth = 5
    ))
])

param_grid = {
    "xgbc__learning_rate" : uniform(0.1 , 0.3),
    "xgbc__gamma" : uniform(0,1), 
    "xgbc__reg_lambda" : uniform(24 , 8),
    "xgbc__reg_alpha" : uniform(0,10),
    "xgbc__min_child_weight" : [2,3,4,5]
}

srch = RandomizedSearchCV(
    estimator = pipeline ,
    param_distributions = param_grid ,
    cv = 5 ,
    n_iter = 15
)

srch.fit(
    X_train ,
    y_train ,
    xgbc__sample_weight = weights
)

model = srch.best_estimator_

print(srch.best_params_)

y_pred = model.predict(X_test)
print(f"F1 SCORE : {f1_score(y_test , y_pred)}")

{'xgbc__gamma': np.float64(0.12354968037551162), 'xgbc__learning_rate': np.float64(0.3510110854726176), 'xgbc__min_child_weight': 3, 'xgbc__reg_alpha': np.float64(3.385633730662371), 'xgbc__reg_lambda': np.float64(28.923555311910665)}
F1 SCORE : 0.9422028353326063


# Final Verdict

# max_depths = 5 
# n_estimators = 374
# min_child_weight = 3
# learning rate = 0.3510
# gamma = 0.1235
# reg_alpha = 3.3856
# reg_lambda = 28.9235